# Figure 7 -- Mechanism (LPS, DNA, safety)

In [ ]:
import pandas as pd,numpy as np,matplotlib.pyplot as plt,matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap,TwoSlopeNorm
from scipy.stats import mannwhitneyu,kruskal
import seaborn as sns,warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-paper')
plt.rcParams.update({
    'font.size':7,'axes.titlesize':7,'axes.labelsize':6.5,
    'xtick.labelsize':6,'ytick.labelsize':6,'legend.fontsize':6,
    'axes.linewidth':0.5,'axes.edgecolor':'#888888',
    'xtick.color':'#555555','ytick.color':'#555555',
    'axes.labelcolor':'#333333','font.family':'sans-serif',
    'font.sans-serif':['Arial','DejaVu Sans'],
    'figure.dpi':300,'savefig.dpi':600,
    'xtick.major.width':0.5,'ytick.major.width':0.5,
    'xtick.major.size':2.5,'ytick.major.size':2.5,
    'xtick.major.pad':2,'ytick.major.pad':2,
    'axes.spines.top':False,'axes.spines.right':False,
})
DATA='/home/pszymczak/code/omegamp-dashboard/data/'
ref=pd.read_csv(DATA+'omegamp_reference_table.csv')
mic=pd.read_csv(DATA+'mic.csv'); hc50=pd.read_csv(DATA+'hc50.csv'); cc50=pd.read_csv(DATA+'cc50.csv')
npn=pd.read_csv(DATA+'npn.csv'); disc=pd.read_csv(DATA+'disc.csv')
lps_raw=pd.read_csv(DATA+'lps_binding.csv'); dna=pd.read_csv(DATA+'dna_binding.csv')

df=ref.merge(hc50,on='short_name',how='left').merge(cc50,on='short_name',how='left')
df=df.merge(npn.rename(columns={'MaxRel':'NPN','AUC':'NPN_AUC'}),on='short_name',how='left')
df=df.merge(disc.rename(columns={'MaxRel':'DiSC','AUC':'DiSC_AUC'}),on='short_name',how='left')
ms=mic.columns[1:]; mv=mic[ms].astype(float)
mic['mic50']=mv.fillna(128).clip(upper=128).median(axis=1)
mic['nle2']=(mv<=2).sum(axis=1); mic['nle4']=(mv<=4).sum(axis=1)
df=df.merge(mic[['short_name','mic50','nle2','nle4']],on='short_name',how='left')
df['HC50']=df['HC50'].clip(0.1,128); df['CC50']=df['CC50'].clip(0.1,128)

pm={'DBAASPS_20015':'OP-145-TII4','DBAASPS_20955':'As-CATH4-6L'}
def gf(r):
    if r['category']=='prototype':
        for f in ['BoCo1','GQ20','Mammutin-1','DeNo1047','cecropin','LG21','pa4','sarcotoxin','bZIP']:
            if f.lower() in r['short_name'].lower() or f==r['short_name']: return f
        if 'OP-145' in r['short_name']: return 'OP-145-TII4'
        if 'As-CATH' in r['short_name']: return 'As-CATH4-6L'
        return r['short_name']
    p=str(r.get('prototype','')); return pm.get(p,p) if p!='nan' else ''
df['family']=df.apply(gf,axis=1)
CL='#0072B2'; CD='#D55E00'
def oc(f): return CD if f=='bZIP' else CL
MO=['cecropin','LG21','pa4','sarcotoxin','bZIP']
LF=['cecropin','LG21','pa4','sarcotoxin']

# Short scatter labels (Omega-num), full heatmap labels
LEADS_SHORT={
    '\u03a9-AMT-cecropin-1':'\u03a9-1','\u03a9-AMT-cecropin-4':'\u03a9-4',
    '\u03a9-AMT-pa4-1':'\u03a9-1','\u03a9-AMT-sarcotoxin-4':'\u03a9-4',
    '\u03a9-AMT-bZIP-8':'\u03a9-8',
}
LEADS_FULL={
    '\u03a9-AMT-cecropin-1':'\u03a9-cecropin-1','\u03a9-AMT-cecropin-4':'\u03a9-cecropin-4',
    '\u03a9-AMT-pa4-1':'\u03a9-pa4-1','\u03a9-AMT-sarcotoxin-4':'\u03a9-sarcotoxin-4',
    '\u03a9-AMT-bZIP-8':'\u03a9-bZIP-8',
}
LS=set(LEADS_SHORT.keys())

# Per-lead offsets: larger to clear scatter clouds
# D: cecropin-1(81.5) cecropin-4(82.4) nearly identical NPN -> big opposing dy
LEAD_OFFSETS_D={
    '\u03a9-AMT-cecropin-1':(8,-16),'\u03a9-AMT-cecropin-4':(8,16),
    '\u03a9-AMT-pa4-1':(8,12),'\u03a9-AMT-sarcotoxin-4':(8,-12),
    '\u03a9-AMT-bZIP-8':(8,-12),
}
# E: pa4-1(104) sarcotoxin-4(101) close DiSC -> opposing dy
LEAD_OFFSETS_E={
    '\u03a9-AMT-cecropin-1':(8,12),'\u03a9-AMT-cecropin-4':(8,-14),
    '\u03a9-AMT-pa4-1':(8,14),'\u03a9-AMT-sarcotoxin-4':(8,-14),
    '\u03a9-AMT-bZIP-8':(8,12),
}

def annotate_lead(ax,x,y,label,offset=(10,8)):
    ax.annotate(label,(x,y),fontsize=6,ha='left',va='center',color='black',
                xytext=offset,textcoords='offset points',
                arrowprops=dict(arrowstyle='-',color='#555',lw=0.4),zorder=10,
                annotation_clip=True)

# NPN/DiSC quadrant colors: greens + purple, colorblind-safe (Okabe-Ito/Tol), distinct from blue/orange of A-C
QC={'NPN dom.':'#009E73','Both strong':'#117733','DiSC dom.':'#AA3377','Both weak':'#BBBBBB'}
QO=['NPN dom.','Both strong','DiSC dom.','Both weak']
me=df.dropna(subset=['NPN','DiSC']).copy()
nm=me['NPN'].median(); dm=me['DiSC'].median()
def aq(r):
    n,d=r['NPN']>=nm,r['DiSC']>=dm
    if n and not d: return 'NPN dom.'
    if not n and d: return 'DiSC dom.'
    if n and d: return 'Both strong'
    return 'Both weak'
me['quad']=me.apply(aq,axis=1)

# Abbreviated strain labels: (display_name, is_mdr, gram)
SF={s:v for s,v in {
    'A. baumannii ATCC 19606 (-)':('Ab. 19606',False,'-'),
    'A. baumannii ATCC BAA-1605 (-) - CGTPACCIMRA':('Ab. BAA (CRAB)',True,'-'),
    'E. cloacae ATCC 13047 (-)':('Ecl. 13047',False,'-'),
    'E. coli ATCC 11775 (-)':('Ec. 11775',False,'-'),
    'E. coli AIC221 (-)':('Ec. AIC221',False,'-'),
    'E. coli AIC222 - CRE (-)':('Ec. AIC222 (CRE)',True,'-'),
    'E. coli ATCC BAA-3170 (-) - CRE':('Ec. BAA (CRE)',True,'-'),
    'K. pneumoniae ATCC 13883 (-)':('Kp. 13883',False,'-'),
    'K. pneumoniae ATCC BAA-2342 (-) - EIRK':('Kp. BAA (ESBL)',True,'-'),
    'P. aeruginosa PAO1 (-)':('Pa. PAO1',False,'-'),
    'P. aeruginosa PA14 (-)':('Pa. PA14',False,'-'),
    'P. aeruginosa ATCC BAA-3197 (-) - FBCRP':('Pa. BAA (FQR)',True,'-'),
    'S. enterica ATCC 9150 (-)':('Se. 9150',False,'-'),
    'S. enterica Typhimurtium ATCC 700720':('Se. Typhimurium',False,'-'),
    'B. subtilis ATCC 23857 (+)':('Bs. 23857',False,'+'),
    'S. aureus ATCC 12600 (+)':('Sa. 12600',False,'+'),
    'S. aureus ATCC BAA-1556 - MRSA (+)':('Sa. BAA (MRSA)',True,'+'),
    'E. faecalis ATCC 700802 - VRE (+)':('Ef. 700802 (VRE)',True,'+'),
    'E. faecium ATCC 700221 - VRE (+)':('Efm. 700221 (VRE)',True,'+'),
    'E. coli K-12 BW25113 (-)':('Ec. K-12',False,'-'),
}.items()}
mic_cmap=LinearSegmentedColormap.from_list('mic_rb',[(0.0,'#b2182b'),(0.25,'#ef8a62'),(0.5,'#f7f7f7'),(0.75,'#67a9cf'),(1.0,'#2166ac')])
mic_norm=TwoSlopeNorm(vmin=0,vcenter=32,vmax=64)
PL=dict(fontsize=9,fontweight='bold',va='top',ha='left')
print(f'Ready: {len(me)}')

In [ ]:
fig=plt.figure(figsize=(6.5,8.0),dpi=300)
# Row layout top-to-bottom: A/B/C | D/E | F/G | H/I | J
# figsize 6.5×8": rows ~1.05" each, gaps: A→D 0.70", D→F 0.35", F→H 0.35", H→J 0.70"
# 0.35" gaps clear 30°-rotated "sarcotoxin" labels at 6pt (~0.30" needed)
gs1      = gridspec.GridSpec(1,3,figure=fig,left=0.10,right=0.97,top=0.963,bottom=0.831,wspace=0.20,width_ratios=[2,2,1.2])
gs2_saf  = gridspec.GridSpec(1,2,figure=fig,left=0.10,right=0.97,top=0.744,bottom=0.613,wspace=0.30)
gs2_mem  = gridspec.GridSpec(1,2,figure=fig,left=0.10,right=0.97,top=0.569,bottom=0.438,wspace=0.30)
gs3_mech = gridspec.GridSpec(1,2,figure=fig,left=0.10,right=0.97,top=0.394,bottom=0.263,wspace=0.50)
gs_hm    = gridspec.GridSpec(1,1,figure=fig,left=0.23,right=0.97,top=0.175,bottom=0.019)

la=lps_raw.groupby(['short_name','family','concentration','calcium'])['BC_displacement'].mean().reset_index()
la=la.merge(ref[['short_name','category']],on='short_name',how='left')
ma=df[(df['family'].isin(MO))&(df['category']=='analog')].copy()

# A
ax=fig.add_subplot(gs1[0]); ax.text(-0.12,1.08,'A',transform=ax.transAxes,**PL)
for i,f in enumerate(MO):
    c=oc(f); fd=df[df['family']==f]; pr=fd[fd['category']=='prototype']; an=fd[fd['category']=='analog']
    if len(pr)>0: ax.scatter(pr['nle2'],i,marker='D',c=c,edgecolors='#333',lw=0.6,s=18,zorder=5)
    if len(an)>0:
        j=np.random.RandomState(42+i).normal(0,0.07,len(an))
        lead_k=0
        for k,(_,r) in enumerate(an.iterrows()):
            if r['short_name'] in LS:
                ax.scatter(r['nle2'],i+j[k],c=c,s=10,edgecolors='black',linewidth=0.5,zorder=5)
                dy=7 if lead_k%2==0 else -7
                annotate_lead(ax,r['nle2'],i+j[k],LEADS_SHORT[r['short_name']],(8,dy))
                lead_k+=1
            else:
                ax.scatter(r['nle2'],i+j[k],c=c,s=8,alpha=0.7,edgecolors='#555',lw=0.2,zorder=4)
ax.set_yticks(range(5)); ax.set_yticklabels([f'$\\it{{{f}}}$' for f in MO],fontsize=6)
ax.invert_yaxis(); ax.set_xlabel('Potent strains (MIC $\\leq$ 2 $\\mu$M)'); ax.set_xlim(-0.5,18)

# B
ax=fig.add_subplot(gs1[1]); ax.text(-0.12,1.08,'B',transform=ax.transAxes,**PL)
for i,f in enumerate(LF):
    fd=la[(la['family']==f)&(la['concentration']==2)&la['calcium'].str.contains('no',case=False,na=False)]
    pr=fd[fd['category']=='prototype']; an=fd[fd['category']!='prototype']
    if len(pr)>0: ax.scatter(pr['BC_displacement'],i,marker='D',c=CL,edgecolors='#333',lw=0.6,s=18,zorder=5)
    if len(an)>0:
        j=np.random.RandomState(42+i).normal(0,0.07,len(an))
        lead_k=0
        for k,(_,r) in enumerate(an.iterrows()):
            if r['short_name'] in LS:
                ax.scatter(r['BC_displacement'],i+j[k],c=CL,s=10,edgecolors='black',linewidth=0.5,zorder=5)
                dy=7 if lead_k%2==0 else -7
                annotate_lead(ax,r['BC_displacement'],i+j[k],LEADS_SHORT[r['short_name']],(8,dy))
                lead_k+=1
            else:
                ax.scatter(r['BC_displacement'],i+j[k],c=CL,s=8,alpha=0.7,edgecolors='#555',lw=0.2,zorder=4)
ax.set_yticks(range(4)); ax.set_yticklabels([f'$\\it{{{f}}}$' for f in LF],fontsize=6)
ax.invert_yaxis(); ax.set_xlabel('LPS binding (BC displacement, %)')

# C
ax=fig.add_subplot(gs1[2]); ax.text(-0.20,1.05,'C',transform=ax.transAxes,**PL)
d12=dna[(dna['metric']=='deltaA')&(dna['concentration']==12.5)].copy()
d12=d12.merge(ref[['short_name','category']],on='short_name',how='left')
d12=d12.sort_values('value',ascending=True).reset_index(drop=True)
y=np.arange(len(d12))
cb_colors=['#bbb' if c=='prototype' else CD for c in d12['category']]
ax.barh(y,d12['value'],color=cb_colors,edgecolor='#666',lw=0.3,height=0.65)
bv=d12[d12['category']=='prototype']['value']
if len(bv)>0: ax.axvline(bv.values[0],color='#bbb',ls=':',lw=0.5)
lb=['bZIP' if c=='prototype' else f"\u03a9-{n.split('-')[-1]}" for c,n in zip(d12['category'],d12['short_name'])]
ax.set_yticks(y); ax.set_yticklabels(lb,fontsize=6)
for i,(_,r) in enumerate(d12.iterrows()):
    if r['short_name'] in LS: ax.get_yticklabels()[i].set_fontweight('bold')
ax.set_xlabel('DNA binding ($\\Delta$A, mdeg)')

fig.legend([Line2D([],[],marker='o',color=CL,lw=0,ms=4,markeredgecolor='#555'),
            Line2D([],[],marker='o',color=CD,lw=0,ms=4,markeredgecolor='#555'),
            Line2D([],[],marker='D',color=CL,markeredgecolor='#333',lw=0,ms=4)],
           ['LPS-binding','DNA-binding','Prototype'],
           loc='upper center',bbox_to_anchor=(0.50,1.00),ncol=3,fontsize=6,frameon=False)

# Helpers
def boxstrip(ax,dd,order,colors):
    for i,k in enumerate(order):
        v=dd.get(k,np.array([]));
        if len(v)==0: continue
        c=colors[k] if isinstance(colors,dict) else colors(k)
        q1,md,q3=np.percentile(v,[25,50,75]); iq=q3-q1
        lo=max(v.min(),q1-1.5*iq); hi=min(v.max(),q3+1.5*iq)
        ax.fill_between([i-0.3,i+0.3],q1,q3,color=c,alpha=0.2,zorder=2)
        ax.plot([i-0.3,i+0.3],[md,md],color='#333',lw=0.8,zorder=4)
        ax.plot([i,i],[lo,q1],color=c,lw=0.6,zorder=2); ax.plot([i,i],[q3,hi],color=c,lw=0.6,zorder=2)
        jt=np.random.RandomState(i+50).normal(0,0.07,len(v))
        ax.scatter(np.full(len(v),i)+jt,v,c=c,s=4,alpha=0.4,edgecolors='#888',lw=0.1,zorder=3)
    ax.set_yscale('log')

def bracket(ax, x1, x2, txt):
    trans=ax.get_xaxis_transform()
    bar_y=0.92; tick_y=0.87
    ax.plot([x1,x1],[tick_y,bar_y],transform=trans,color='#444',lw=0.6,clip_on=False)
    ax.plot([x2,x2],[tick_y,bar_y],transform=trans,color='#444',lw=0.6,clip_on=False)
    ax.plot([x1,x2],[bar_y,bar_y],transform=trans,color='#444',lw=0.6,clip_on=False)
    ax.text(0.5,1.05,txt,transform=ax.transAxes,ha='center',va='bottom',fontsize=6,color='#888',style='italic')

# D — HC50 by family
ax=fig.add_subplot(gs2_saf[0]); ax.text(-0.16,1.05,'D',transform=ax.transAxes,**PL)
hcd={f:ma[ma['family']==f]['HC50'].dropna().values for f in MO}
boxstrip(ax,hcd,MO,{f:oc(f) for f in MO})
for i,f in enumerate(MO):
    pr=df[(df['family']==f)&(df['category']=='prototype')]
    if len(pr)>0 and pd.notna(pr['HC50'].values[0]): ax.scatter([i],pr['HC50'].values[0],marker='D',c=oc(f),edgecolors='#333',lw=0.6,s=18,zorder=5)
ax.set_xticks(range(5)); ax.set_xticklabels([f'$\\it{{{f}}}$' for f in MO],fontsize=6,rotation=30,ha='right')
ax.set_ylabel('HC$_{50}$ ($\\mu$M)')
gs2_=[v for v in hcd.values() if len(v)>0]
if len(gs2_)>=2:
    _,pk=kruskal(*gs2_)
    ax.text(0.5,1.05,f'KW p = {pk:.0e}',transform=ax.transAxes,fontsize=6,ha='center',va='bottom',color='#888',style='italic')

# E — CC50 by family
ax=fig.add_subplot(gs2_saf[1]); ax.text(-0.16,1.05,'E',transform=ax.transAxes,**PL)
ccd={f:ma[ma['family']==f]['CC50'].dropna().values for f in MO}
boxstrip(ax,ccd,MO,{f:oc(f) for f in MO})
for i,f in enumerate(MO):
    pr=df[(df['family']==f)&(df['category']=='prototype')]
    if len(pr)>0 and pd.notna(pr['CC50'].values[0]): ax.scatter([i],pr['CC50'].values[0],marker='D',c=oc(f),edgecolors='#333',lw=0.6,s=18,zorder=5)
ax.set_xticks(range(5)); ax.set_xticklabels([f'$\\it{{{f}}}$' for f in MO],fontsize=6,rotation=30,ha='right')
ax.set_ylabel('CC$_{50}$ ($\\mu$M)')

# F — NPN MaxRel by family
mm=me[me['family'].isin(MO)].copy()
ax=fig.add_subplot(gs2_mem[0]); ax.text(-0.08,1.05,'F',transform=ax.transAxes,**PL)
for j_idx,f in enumerate(MO):
    c=oc(f); an=mm[(mm['family']==f)&(mm['category']=='analog')]
    pr=mm[(mm['family']==f)&(mm['category']=='prototype')]
    jt=np.random.RandomState(j_idx*7).normal(0,0.1,len(an)) if len(an)>0 else []
    for k,(_,r) in enumerate(an.iterrows()):
        if r['short_name'] in LS and pd.notna(r['NPN']):
            ax.scatter(j_idx+jt[k],r['NPN'],c=c,s=10,edgecolors='black',linewidth=0.5,zorder=5)
            off=LEAD_OFFSETS_D.get(r['short_name'],(6,8))
            annotate_lead(ax,j_idx+jt[k],r['NPN'],LEADS_SHORT[r['short_name']],off)
        elif pd.notna(r['NPN']):
            ax.scatter(j_idx+jt[k],r['NPN'],c=c,s=7,alpha=0.7,edgecolors='#555',lw=0.2,zorder=3)
    if len(pr)>0:
        pv=pr['NPN'].dropna().values
        if len(pv)>0: ax.scatter([j_idx],pv,marker='D',c=c,edgecolors='#333',lw=0.6,s=18,zorder=5)
ax.axhline(nm,color='#ddd',ls=':',lw=0.5,zorder=1)
ax.set_xticks(range(5)); ax.set_xticklabels([f'$\\it{{{f}}}$' for f in MO],fontsize=6,rotation=30,ha='right')
ax.set_ylabel('NPN MaxRel (%)')
gs_npn=[mm[(mm['family']==f)&(mm['category']=='analog')]['NPN'].dropna().values for f in MO]
gs_npn=[g for g in gs_npn if len(g)>0]
if len(gs_npn)>=2:
    av=np.concatenate(gs_npn); gm=av.mean()
    ax.text(0.97,0.97,f'\u03b7\u00b2 = {sum(len(g)*(g.mean()-gm)**2 for g in gs_npn)/((av-gm)**2).sum():.2f}',
            transform=ax.transAxes,fontsize=6,ha='right',va='top',color='#555')

# G — DiSC MaxRel by family
ax=fig.add_subplot(gs2_mem[1]); ax.text(-0.08,1.05,'G',transform=ax.transAxes,**PL)
for j_idx,f in enumerate(MO):
    c=oc(f); an=mm[(mm['family']==f)&(mm['category']=='analog')]
    pr=mm[(mm['family']==f)&(mm['category']=='prototype')]
    jt=np.random.RandomState(j_idx*7+1).normal(0,0.1,len(an)) if len(an)>0 else []
    for k,(_,r) in enumerate(an.iterrows()):
        if r['short_name'] in LS and pd.notna(r['DiSC']):
            ax.scatter(j_idx+jt[k],r['DiSC'],c=c,s=10,edgecolors='black',linewidth=0.5,zorder=5)
            off=LEAD_OFFSETS_E.get(r['short_name'],(6,8))
            annotate_lead(ax,j_idx+jt[k],r['DiSC'],LEADS_SHORT[r['short_name']],off)
        elif pd.notna(r['DiSC']):
            ax.scatter(j_idx+jt[k],r['DiSC'],c=c,s=7,alpha=0.7,edgecolors='#555',lw=0.2,zorder=3)
    if len(pr)>0:
        pv=pr['DiSC'].dropna().values
        if len(pv)>0: ax.scatter([j_idx],pv,marker='D',c=c,edgecolors='#333',lw=0.6,s=18,zorder=5)
ax.axhline(dm,color='#ddd',ls=':',lw=0.5,zorder=1)
ax.set_xticks(range(5)); ax.set_xticklabels([f'$\\it{{{f}}}$' for f in MO],fontsize=6,rotation=30,ha='right')
ax.set_ylabel('DiSC$_3$(5) MaxRel (%)',labelpad=1)
gs_disc=[mm[(mm['family']==f)&(mm['category']=='analog')]['DiSC'].dropna().values for f in MO]
gs_disc=[g for g in gs_disc if len(g)>0]
if len(gs_disc)>=2:
    av=np.concatenate(gs_disc); gm=av.mean()
    ax.text(0.97,0.97,f'\u03b7\u00b2 = {sum(len(g)*(g.mean()-gm)**2 for g in gs_disc)/((av-gm)**2).sum():.2f}',
            transform=ax.transAxes,fontsize=6,ha='right',va='top',color='#555')

# H — HC50 by membrane mechanism
ax=fig.add_subplot(gs3_mech[0]); ax.text(-0.16,1.05,'H',transform=ax.transAxes,**PL)
mh=me.dropna(subset=['HC50']); hq={q:mh[mh['quad']==q]['HC50'].values for q in QO}
boxstrip(ax,hq,QO,QC)
ax.set_xticks(range(4)); ax.set_xticklabels(['NPN\ndom.','Both\nstrong','DiSC\ndom.','Both\nweak'],fontsize=6)
ax.set_ylabel('HC$_{50}$ ($\\mu$M)')
nh=hq.get('NPN dom.',[]); dh=hq.get('DiSC dom.',[])
if len(nh)>0 and len(dh)>0:
    _,p=mannwhitneyu(nh,dh)
    bracket(ax, 0, 2, f'p = {p:.0e}')

# I — CC50 by membrane mechanism
ax=fig.add_subplot(gs3_mech[1]); ax.text(-0.16,1.05,'I',transform=ax.transAxes,**PL)
mc=me.dropna(subset=['CC50']); cq={q:mc[mc['quad']==q]['CC50'].values for q in QO}
boxstrip(ax,cq,QO,QC)
ax.set_xticks(range(4)); ax.set_xticklabels(['NPN\ndom.','Both\nstrong','DiSC\ndom.','Both\nweak'],fontsize=6)
ax.set_ylabel('CC$_{50}$ ($\\mu$M)')
nc=cq.get('NPN dom.',[]); dc=cq.get('DiSC dom.',[])
if len(nc)>0 and len(dc)>0:
    _,p=mannwhitneyu(nc,dc)
    bracket(ax, 0, 2, f'p = {p:.0e}')

# Legend for H/I — in the 0.70" gap between H/I bottom (0.263) and J top (0.175)
le=[Line2D([],[],marker='o',color=QC[q],lw=0,ms=3.5,markeredgecolor='#888',label=q.replace(' dom.',' dominant')) for q in QO]
fig.legend(handles=le,loc='lower center',ncol=4,fontsize=6,frameon=False,bbox_to_anchor=(0.52,0.22))

# J — MIC heatmap
ax=fig.add_subplot(gs_hm[0])
fig.text(0.08, 0.19, 'J', **PL)
BEST_LEADS={'cecropin':'\u03a9-AMT-cecropin-1','LG21':None,'pa4':'\u03a9-AMT-pa4-1',
            'sarcotoxin':'\u03a9-AMT-sarcotoxin-4','bZIP':'\u03a9-AMT-bZIP-8'}
so=sorted(ms,key=lambda s:(0 if '(+)' in s else 1,0 if any(t in s for t in ['CRAB','CRE','EIRK','ESBL','FQR','FBCR','MRSA','VRE']) else 1,s))[::-1]
sl=[SF.get(s,(s[:12],False,'-'))[0] for s in so]
rows_data,row_labels=[],[]
for f in MO:
    proto_sn=df[(df['family']==f)&(df['category']=='prototype')]['short_name'].values
    proto_mic_row=mic[mic['short_name'].isin(proto_sn)]
    if len(proto_mic_row)>0:
        rows_data.append({s:float(proto_mic_row[s].values[0]) for s in so}); row_labels.append(f'$\\it{{{f}}}$')
    lead_sn=BEST_LEADS.get(f)
    if lead_sn and lead_sn in mic['short_name'].values: lead_row=mic[mic['short_name']==lead_sn]
    else:
        fam_ana=ma[ma['family']==f].dropna(subset=['mic50'])
        if len(fam_ana)>0: best_sn=fam_ana.sort_values('mic50').iloc[0]['short_name']; lead_row=mic[mic['short_name']==best_sn]; lead_sn=best_sn
        else: lead_row=pd.DataFrame()
    if len(lead_row)>0:
        rows_data.append({s:float(lead_row[s].values[0]) for s in so})
        if lead_sn in LS:
            row_labels.append(f'$\\bf{{{LEADS_FULL[lead_sn]}}}$')
        else: row_labels.append(lead_sn.replace('\u03a9-AMT-','\u03a9-'))
hm=pd.DataFrame(rows_data); hm.columns=sl; hm.index=row_labels
mic_norm_hm=TwoSlopeNorm(vmin=0,vcenter=32,vmax=64)
annot_str=np.empty_like(hm.values,dtype=object)
for i in range(hm.values.shape[0]):
    for j in range(hm.values.shape[1]):
        v=hm.values[i,j]
        if np.isnan(v) or v>64: annot_str[i,j]=''
        elif v<1: annot_str[i,j]=f'{v:.1f}'
        else: annot_str[i,j]=str(int(v))
hm_clip=hm.clip(0,64).fillna(64)
sns.heatmap(hm_clip,ax=ax,cmap=mic_cmap,norm=mic_norm_hm,linewidths=0.4,linecolor='white',
            xticklabels=True,yticklabels=True,
            cbar_kws={'label':'MIC ($\\mu$M)','shrink':0.7,'pad':0.01,'ticks':[0,8,16,32,48,64]},
            annot=annot_str,fmt='',annot_kws={'fontsize':7})
for t in ax.texts:
    try:
        v=float(t.get_text()); normed=mic_norm_hm(min(v,64))
        t.set_color('white' if normed<0.15 or normed>0.85 else 'black')
    except: pass
for i in range(2,len(row_labels),2): ax.axhline(y=i,color='#888',lw=0.6,zorder=5)
xlabels=ax.get_xticklabels()
for i,s in enumerate(so):
    info=SF.get(s,('',False,'-')); txt=xlabels[i]
    txt.set_rotation(45); txt.set_ha('right'); txt.set_fontsize(6); txt.set_fontstyle('italic')
    txt.set_color('#DC2626' if info[2]=='+' else '#2563EB')
    if info[1]: txt.set_fontweight('bold')

plt.savefig('/home/pszymczak/code/omegamp-dashboard/figures/figure6_mechanism.pdf',bbox_inches='tight')
plt.savefig('/home/pszymczak/code/omegamp-dashboard/figures/figure6_mechanism.svg',bbox_inches='tight')
plt.savefig('/home/pszymczak/code/omegamp-dashboard/figures/figure6_mechanism.png',dpi=600,bbox_inches='tight')
plt.show()
print('Saved figure6_mechanism')